In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import col

# Streaming view from bronze — casts STRING columns to proper types
# Filters out records with no valid position (empty strings cast to null)
@dp.view
def flights_bronze_stream():
    return (
        spark.readStream.table("dbr_dev.skybound_bronze.flights_bronze")
        .select(
            col("icao24"),
            col("callsign"),
            col("origin_country"),
            col("longitude").cast("double"),
            col("latitude").cast("double"),
            col("geo_altitude").cast("double").alias("altitude"),
            col("velocity").cast("double").alias("speed"),
            col("true_track").cast("double").alias("heading"),
            col("vertical_rate").cast("double"),
            col("on_ground").cast("boolean"),
            col("ingestion_timestamp").cast("timestamp").alias("last_seen")
        )
        .where("latitude IS NOT NULL AND longitude IS NOT NULL")
    )

In [0]:
# Target streaming table — one row per aircraft (SCD Type 1)
dp.create_streaming_table(
    name="flights_silver",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.zOrderCols": "icao24"
    }
)

# SCD Type 1: keep only the latest position per aircraft
# Handles out-of-order events automatically via sequence_by
dp.create_auto_cdc_flow(
    target="flights_silver",
    source="flights_bronze_stream",
    keys=["icao24"],
    sequence_by=col("last_seen"),
    stored_as_scd_type=1
)